# SI Lambda Sweep
Sweeps `si_lambda` (called `c` in the SI literature) over `[50, 100, 250, 500, 1000, 2000]` on the `axis1_delta1.5_sudden` dataset.

Each task period is split **64 / 16 / 20** (train / val / test). SMOTE is applied to training only.
The best lambda is selected by validation AvgPR and then re-evaluated on the held-out test set.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import FEATURE_COLS
from src.methods import FraudDetector, SynapticIntelligence, train_with_si
from src.utils import compute_thesis_metrics, set_seed, test_on_task

with open(PROJECT_ROOT / 'config/experiment.yaml') as f:
    cfg = yaml.safe_load(f)

print('Setup complete. Project root:', PROJECT_ROOT)

In [16]:
LAMBDAS       = [50, 100, 250, 500, 1000, 2000]
DATASET_PATH  = PROJECT_ROOT / cfg['paths']['base_dir'] / 'axis1_delta1.5_sudden.csv'
N_TASKS       = cfg['n_tasks']
N_EPOCHS      = cfg['n_epochs']
LR            = cfg['learning_rate']
SI_XI         = cfg['si_xi']
SEED          = cfg['seed']

In [17]:
def get_task_splits(df, task_id, scaler=None, seed=42):
    """Return (X_train, y_train), (X_val, y_val), (X_test, y_test), scaler for one task.

    Split is 64 / 16 / 20 (train / val / test) via two successive 80/20 splits.
    SMOTE is applied to the training fold only.
    Scaler is fit once on the first task's training data and must be passed back
    in for subsequent tasks to avoid leakage.
    """
    period = task_id + 1
    period_df = df[df['period'] == period]
    if len(period_df) == 0:
        raise ValueError(f"Period {period} not found in dataset.")

    X_all = period_df[FEATURE_COLS].values.astype('float32')
    y_all = period_df['is_fraud'].values.astype('float32')

    n_pos, n_neg = int((y_all == 1).sum()), int((y_all == 0).sum())
    use_strat = (n_pos >= 2) and (n_neg >= 2)

    # First split: 80 % temp + 20 % test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_all, y_all, test_size=0.20, random_state=seed,
        stratify=y_all if use_strat else None,
    )

    # Second split: 80 % of temp \u2192 train (64 % total) + val (16 % total)
    n_pos_t = int((y_temp == 1).sum())
    n_neg_t = int((y_temp == 0).sum())
    use_strat2 = (n_pos_t >= 2) and (n_neg_t >= 2)

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.20, random_state=seed,
        stratify=y_temp if use_strat2 else None,
    )

    # Scaler: fit on first task's train split, reuse for all later tasks
    if scaler is None:
        scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    # SMOTE on training fold only
    n_pos_train = int((y_train == 1).sum())
    if n_pos_train >= 6:
        smote = SMOTE(k_neighbors=5, sampling_strategy='auto', random_state=seed)
        X_train, y_train = smote.fit_resample(X_train, y_train)

    def to_t(X, y):
        return torch.FloatTensor(X), torch.FloatTensor(y).unsqueeze(1)

    return to_t(X_train, y_train), to_t(X_val, y_val), to_t(X_test, y_test), scaler

In [18]:
def run_si_sweep_with_val(dataset_path, lambdas, n_tasks, n_epochs, lr, si_xi, seed=42):
    """Sweep SI lambda (c) using a validation set; report test performance for the best lambda.

    For each lambda:
      - Pre-computed train/val/test splits are reused (deterministic, same seed).
      - Model trains on train splits; val PR-AUC matrix drives lambda selection.
    After the sweep, the best lambda (highest val AvgPR) is retrained and evaluated on test.

    Returns
    -------
    val_results : dict[lambda -> dict]   per-lambda val metrics
    best_lam    : int/float              lambda with highest val AvgPR
    test_result : dict                   test metrics for best_lam
    """
    df = pd.read_csv(dataset_path)
    print(f"Loaded {len(df):,} rows | fraud rate: {df['is_fraud'].mean()*100:.3f}%\n")

    # Pre-compute all splits once \u2014 they are deterministic so all lambdas share them
    print("Pre-computing 64/16/20 splits for all tasks...")
    splits = []
    scaler = None
    for task_id in range(n_tasks):
        train_data, val_data, test_data, scaler = get_task_splits(df, task_id, scaler, seed)
        splits.append((train_data, val_data, test_data))
    print(f"  Done. {n_tasks} tasks \u00d7 3 splits ready.\n")

    val_results = {}

    for lam in lambdas:
        print(f'  lambda={lam}', end=' ... ')
        set_seed(seed)
        model = FraudDetector(input_features=cfg['n_features'])
        si    = SynapticIntelligence(xi=si_xi)
        val_pr = np.zeros((n_tasks, n_tasks))

        for task_id in range(n_tasks):
            (X_train, y_train), _, _ = splits[task_id]
            model, small_omega, prev_params = train_with_si(
                model, X_train, y_train, si,
                importance_factor=lam, epochs=n_epochs, lr=lr)
            si.register_task(model, small_omega, prev_params)

            for val_id in range(task_id + 1):
                _, (X_v, y_v), _ = splits[val_id]
                val_pr[task_id, val_id] = test_on_task(model, X_v, y_v)['pr_auc']

        _, _, avg_pr, forget_pr, diag_pr = compute_thesis_metrics(np.zeros_like(val_pr), val_pr)
        val_results[lam] = {'avg_pr': avg_pr, 'forget_pr': forget_pr, 'diag_pr': diag_pr}
        print(f'[VAL] AvgPR={avg_pr:.4f} | ForgetPR={forget_pr:.4f} | DiagPR={diag_pr:.4f}')

    # Select best lambda by validation AvgPR
    best_lam = max(val_results, key=lambda l: val_results[l]['avg_pr'])
    print(f'\nBest lambda (val AvgPR): lambda={best_lam}  \u2192  AvgPR={val_results[best_lam]["avg_pr"]:.4f}')

    # Retrain with best lambda and evaluate on the held-out test splits
    print(f'\nRetraining with lambda={best_lam} \u2014 evaluating on test set...')
    set_seed(seed)
    model = FraudDetector(input_features=cfg['n_features'])
    si    = SynapticIntelligence(xi=si_xi)
    test_pr = np.zeros((n_tasks, n_tasks))

    for task_id in range(n_tasks):
        (X_train, y_train), _, _ = splits[task_id]
        model, small_omega, prev_params = train_with_si(
            model, X_train, y_train, si,
            importance_factor=best_lam, epochs=n_epochs, lr=lr)
        si.register_task(model, small_omega, prev_params)

        for test_id in range(task_id + 1):
            _, _, (X_t, y_t) = splits[test_id]
            test_pr[task_id, test_id] = test_on_task(model, X_t, y_t)['pr_auc']

    _, _, avg_pr_t, forget_pr_t, diag_pr_t = compute_thesis_metrics(np.zeros_like(test_pr), test_pr)
    test_result = {'avg_pr': avg_pr_t, 'forget_pr': forget_pr_t, 'diag_pr': diag_pr_t}
    print(f'[TEST] AvgPR={avg_pr_t:.4f} | ForgetPR={forget_pr_t:.4f} | DiagPR={diag_pr_t:.4f}')

    return val_results, best_lam, test_result

In [ ]:
val_results, best_lam, test_result = run_si_sweep_with_val(
    DATASET_PATH, LAMBDAS, N_TASKS, N_EPOCHS, LR, SI_XI, seed=SEED
)

In [ ]:
METRICS = [('avg_pr', 'AvgPR'), ('forget_pr', 'ForgetPR')]

mat_val = np.array([[val_results[lam][key] for key, _ in METRICS] for lam in LAMBDAS])

# Normalize each column to [0, 1] where 1 = best, so RdYlGn is always intuitive
# AvgPR: higher is better  →  normalize directly
# ForgetPR: lower is better →  invert normalization
def col_normalize(col, higher_is_better):
    mn, mx = col.min(), col.max()
    if mx == mn:
        return np.full_like(col, 0.5)
    norm = (col - mn) / (mx - mn)
    return norm if higher_is_better else 1 - norm

mat_display = np.column_stack([
    col_normalize(mat_val[:, 0], higher_is_better=True),
    col_normalize(mat_val[:, 1], higher_is_better=False),
])

fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle('SI Lambda Sweep — Sudden: Delta 1.5', fontsize=13, fontweight='bold')

im = ax.imshow(mat_display, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

for r in range(len(LAMBDAS)):
    for c in range(len(METRICS)):
        ax.text(c, r, f'{mat_val[r, c]:.4f}', ha='center', va='center',
                fontsize=11, fontweight='bold', color='black')

ax.set_title('Validation (all lambdas)', fontweight='bold')
ax.set_xticks(range(len(METRICS)))
ax.set_xticklabels([label for _, label in METRICS], fontsize=11)
ax.set_yticks(range(len(LAMBDAS)))
ax.set_yticklabels([f'λ={lam}' + (' ★' if lam == best_lam else '') for lam in LAMBDAS], fontsize=10)

plt.tight_layout()
out = PROJECT_ROOT / cfg['paths']['figures_dir']
out.mkdir(parents=True, exist_ok=True)
plt.savefig(out / 'si_lambda_sweep_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def _pareto_optimal(points):
    """Boolean mask: True if no other point dominates (maximise both objectives)."""
    n = len(points)
    is_opt = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i != j and (points[j] >= points[i]).all() and (points[j] > points[i]).any():
                is_opt[i] = False
                break
    return is_opt

forget_vals = np.array([val_results[lam]['forget_pr'] for lam in LAMBDAS])
avg_vals    = np.array([val_results[lam]['avg_pr']    for lam in LAMBDAS])
lambdas_arr = np.array(LAMBDAS)

# ForgetPR: lower is better (positive = forgetting, negative = backward transfer)
# Negate ForgetPR so the Pareto function can maximise both transformed objectives
pts_pareto = np.column_stack([-forget_vals, avg_vals])
is_pareto  = _pareto_optimal(pts_pareto)

# Sort Pareto front by ForgetPR ascending (lower = better = left side of plot)
front_idx = np.where(is_pareto)[0]
front_idx = front_idx[np.argsort(forget_vals[front_idx])]

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(forget_vals[~is_pareto], avg_vals[~is_pareto],
           color='steelblue', s=80, alpha=0.7, zorder=3, label='Dominated')
ax.scatter(forget_vals[is_pareto], avg_vals[is_pareto],
           color='crimson', s=160, marker='*', zorder=4, label='Pareto-optimal')
ax.plot(forget_vals[front_idx], avg_vals[front_idx],
        color='crimson', lw=1.2, ls='--', alpha=0.55, zorder=2)

for i, lam in enumerate(lambdas_arr):
    ax.annotate(f'λ={lam}', (forget_vals[i], avg_vals[i]),
                textcoords='offset points', xytext=(6, 4), fontsize=10)

ax.set_xlabel('ForgetPR  (↓ less forgetting)', fontsize=12)
ax.set_ylabel('AvgPR  (↑ better overall performance)', fontsize=12)
ax.set_title('Pareto Front — SI λ sweep (validation) — Sudden: Delta 1.5', fontsize=10, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, ls='--', alpha=0.4)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / cfg['paths']['figures_dir'] / 'si_pareto_front.png', dpi=150, bbox_inches='tight')
plt.show()

print('Pareto-optimal lambdas:', lambdas_arr[is_pareto].tolist())
